# 04 · Retrieve — 05 Ranking and `final_score`

**Everything in this notebook is offline and deterministic: synthetic candidate papers, zero network calls, zero API calls -- including the re-ranking demonstration.**

Implements `quality_weighted_sort` (a top-level demo function here, rather
than a private helper) and the `journal_quality_score` lookup.

**In → out:** the RCS-scored survivors from notebook `04` → the same papers,
each carrying a `final_score` and a persisted `final_score_components` dict --
five normalized signals (relevance, citations, recency, journal impact,
evidence level) blended by a configurable weight vector.

**The property this notebook exists to prove:** because all five components
are persisted per paper, a *different* ranking can be re-derived later from
those stored numbers alone -- no re-retrieval, no re-scoring, no model call.
An ablation is just a different weight vector applied to numbers you already
have. Each of the five components below is computed and looked at alone,
against the same synthetic candidates, before they're blended together.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `journal_quality_score` | Looks up a journal's quality in a CSV, `-1` if missing (known defect 1 below). | `journal_quality_score("Nature", journal_quality_map)` |
| relevance component | `rcs_score` (or vector `score`) / 10, clamped to `[0, 1]`. | one normalized value per candidate |
| citations component | `log1p(citation_count)`, scaled to the batch's own max. | one normalized value per candidate |
| recency component | linear decay from 1.0 (this year) to 0.0 at `max_age_years`. | one normalized value per candidate |
| journal_impact component | raw `journal_quality_score` / 5, clamped -- `-1` clamps to `0.0`. | one normalized value per candidate |
| evidence_level component | a fixed Oxford CEBM map, Level I → 1.0 down to Level V → 0.0. | one normalized value per candidate |
| `quality_weighted_sort` | Blends all five normalized components with a weight vector into `final_score`, and persists `final_score_components`. | `quality_weighted_sort(candidates, DEFAULT_WEIGHTS, quality_map=journal_quality_map)` |
| `rerank_from_components` | Re-derives a ranking from already-persisted components and a *new* weight vector -- no re-retrieval. | `rerank_from_components(ranked, RECENCY_FIRST_WEIGHTS)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — weights are configuration, not code

In a real deployment, these weights would come from a domain-specific
config file (e.g. per clinical department or per course). No department
config, course config, or partner data travels into this repo --
`weights.json`, next to this notebook, ships one neutral, illustrative
default weight set instead (the same fallback numbers used when nothing
domain-specific is configured). Load it from disk rather than hardcoding it
here, so it's genuinely one file a reader could replace.

In [ ]:
import json

weights_path = Path.cwd() / "weights.json"
DEFAULT_WEIGHTS = json.loads(weights_path.read_text())
nbio.show_json(DEFAULT_WEIGHTS)

## Step 2 — the journal-quality signal, and known defect 1

`journal_quality_score` looks a journal name up in a CSV
(`clean_name,quality`) and returns `-1` if the journal isn't found -- or if
the CSV isn't there at all. **In the real product, the CSV isn't there at
all**: the configured path points into a vendored `paper-qa` checkout that
was never added to the repo, so every paper scores `-1`, and the
`journal_impact` component normalizes to `0.00` for every paper in every
ranking -- one of five signals, silently contributing nothing.

This repo ships `data/journal_quality_example.csv`, a **tiny, illustrative**
ten-journal example -- explicitly not the real PaperQA2 dataset -- so the
signal below can be shown actually contributing something. The cell after it
simulates the real product's missing-file case side by side, so you can see
the exact degraded state directly rather than take the claim on faith.

In [ ]:
import csv


def load_journal_quality_map(csv_path: Path) -> dict[str, int]:
    if not csv_path.exists():
        return {}
    data: dict[str, int] = {}
    with csv_path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            key = (row.get("clean_name") or "").strip()
            if not key:
                continue
            try:
                data[key] = int((row.get("quality") or "").strip())
            except ValueError:
                data[key] = -1
    return data


def journal_quality_score(journal: str | None, quality_map: dict[str, int]) -> int:
    if not journal or not str(journal).strip():
        return -1
    return quality_map.get(str(journal).casefold().replace("amp;", ""), -1)


journal_quality_map = load_journal_quality_map(Path.cwd() / "data" / "journal_quality_example.csv")
print(f"loaded {len(journal_quality_map)} journals from the illustrative example CSV\n")

for j in ["Nature", "The Lancet", "PLOS ONE", "A Regional Wound Care Newsletter"]:
    print(f"  {j:35s} -> quality={journal_quality_score(j, journal_quality_map)}")

## Step 3 — synthetic candidate papers (anchor data)

Eight invented papers -- no real bibliographic data -- with a spread of
citation counts, ages, journals (some in the tiny example CSV, some not),
and evidence levels, standing in for what would be the RCS-scored survivors
coming out of notebook `04`. Defined here, before any of the five scoring
components below, so each component has real data to compute against.

In [ ]:
import datetime

this_year = datetime.datetime.now().year

candidates = [
    {"title": "RCT of NPWT in diabetic foot ulcers", "rcs_score": 9, "citation_count": 240,
     "year": this_year - 3, "journal": "The Lancet", "evidence_level": "Level I"},
    {"title": "Meta-analysis of amputation risk reduction strategies", "rcs_score": 8, "citation_count": 512,
     "year": this_year - 6, "journal": "Diabetes Care", "evidence_level": "Level I"},
    {"title": "Cohort study of NPWT vs standard dressings", "rcs_score": 7, "citation_count": 34,
     "year": this_year - 1, "journal": "Wound Repair and Regeneration", "evidence_level": "Level II"},
    {"title": "Case series on pressure injury staging", "rcs_score": 6, "citation_count": 5,
     "year": this_year - 9, "journal": "PLOS ONE", "evidence_level": "Level IV"},
    {"title": "Expert consensus on offloading techniques", "rcs_score": 6, "citation_count": 18,
     "year": this_year - 4, "journal": "A Regional Wound Care Newsletter", "evidence_level": "Level V"},
    {"title": "Retrospective review of NPWT adoption patterns", "rcs_score": 5, "citation_count": 90,
     "year": this_year - 12, "journal": "Nature Medicine", "evidence_level": "Level III"},
    {"title": "Narrative review of diabetic foot ulcer epidemiology", "rcs_score": 5, "citation_count": 3,
     "year": this_year, "journal": "BMJ", "evidence_level": "Unknown"},
    {"title": "Single-arm pilot study of a novel dressing", "rcs_score": 6, "citation_count": 1,
     "year": this_year - 2, "journal": "A Regional Wound Care Newsletter", "evidence_level": "Level IV"},
]

print(f"{len(candidates)} synthetic candidate papers")

## Step 4 — the relevance component, alone

`rcs_score` (or a vector `score`) divided by 10, clamped to `[0, 1]`. Computed
and looked at on its own, against the real candidates above, before any
other component exists.

In [ ]:
rel_norm = [max(0.0, min(1.0, float(d.get("rcs_score") or d.get("score") or 0.0) / 10.0)) for d in candidates]
for d, r in zip(candidates, rel_norm):
    print(f"  {r:.3f}  {d['title'][:55]}")

## Step 5 — the citations component, alone

`log1p(citation_count)`, then divided by the batch's own max `log1p` value --
so this component is relative to the candidate set, not an absolute scale.

In [ ]:
import math

cit_log = [math.log1p(max(0, int(d.get("citation_count") or 0))) for d in candidates]
cit_max = max(cit_log) if cit_log else 0.0
cit_norm = [(v / cit_max) if cit_max > 0 else 0.0 for v in cit_log]
for d, c in zip(candidates, cit_norm):
    print(f"  {c:.3f}  {d['title'][:55]}")

## Step 6 — the recency component, alone

Linear decay from 1.0 (this year) to 0.0 at `max_age_years` years old
(default 10).

In [ ]:
max_age_years = 10
rec_norm = []
for d in candidates:
    y = int(d.get("year") or 0)
    if y <= 0:
        rec_norm.append(0.0)
        continue
    age = max(0, min(max_age_years, this_year - y))
    rec_norm.append(1.0 - (age / max_age_years) if max_age_years > 0 else 0.0)
for d, rc in zip(candidates, rec_norm):
    print(f"  {rc:.3f}  {d['title'][:55]}")

## Step 7 — the journal_impact component, alone

Raw `journal_quality_score` (0-5 scale, or -1 if unknown) divided by 5,
clamped to `[0, 1]` -- so `-1` clamps to `0.0`, exactly the defect described
in Step 2. Reuses `journal_quality_score` and `journal_quality_map` from
Step 2, on the real candidates.

In [ ]:
jq_raw = [journal_quality_score(d.get("journal"), journal_quality_map) for d in candidates]
jq_norm = [max(0.0, min(1.0, v / 5.0)) for v in jq_raw]
for d, jq in zip(candidates, jq_norm):
    print(f"  {jq:.3f}  {d['title'][:55]}")

## Step 8 — the evidence_level component, alone

A fixed Oxford CEBM map, Level I → 1.0 down to Level V → 0.0, Unknown →
0.25.

In [ ]:
_EVIDENCE_LEVEL_SCORE = {
    "Level I": 1.0,
    "Level II": 0.75,
    "Level III": 0.50,
    "Level IV": 0.25,
    "Level V": 0.0,
    "Unknown": 0.25,
}

ev_raw = [_EVIDENCE_LEVEL_SCORE.get(str(d.get("evidence_level") or "Unknown"), 0.25) for d in candidates]
for d, ev in zip(candidates, ev_raw):
    print(f"  {ev:.3f}  {d['title'][:55]}")

## Step 9 — blend all five into `quality_weighted_sort`

`final_score` is the weighted sum of the five components computed
individually above -- this function recomputes them the same way, over any
`docs` list, and is -- this is the part worth keeping -- `final_score_components`
is attached to every paper alongside it, not discarded once the sort
finishes.

In [ ]:
def quality_weighted_sort(docs: list[dict], weights: dict, *, quality_map: dict, max_age_years: int = 10) -> list[dict]:
    if not docs:
        return docs

    w_rel = weights.get("relevance", 0.45)
    w_cit = weights.get("citations", 0.20)
    w_rec = weights.get("recency", 0.15)
    w_jq = weights.get("journal_impact", 0.10)
    w_ev = weights.get("evidence_level", 0.10)
    now_year = datetime.datetime.now().year

    rel_norm = [max(0.0, min(1.0, float(d.get("rcs_score") or d.get("score") or 0.0) / 10.0)) for d in docs]

    cit_log = [math.log1p(max(0, int(d.get("citation_count") or 0))) for d in docs]
    cit_max = max(cit_log) if cit_log else 0.0
    cit_norm = [(v / cit_max) if cit_max > 0 else 0.0 for v in cit_log]

    rec_norm = []
    for d in docs:
        y = int(d.get("year") or 0)
        if y <= 0:
            rec_norm.append(0.0)
            continue
        age = max(0, min(max_age_years, now_year - y))
        rec_norm.append(1.0 - (age / max_age_years) if max_age_years > 0 else 0.0)

    jq_raw = [journal_quality_score(d.get("journal"), quality_map) for d in docs]
    jq_norm = [max(0.0, min(1.0, v / 5.0)) for v in jq_raw]

    ev_raw = [_EVIDENCE_LEVEL_SCORE.get(str(d.get("evidence_level") or "Unknown"), 0.25) for d in docs]

    out = []
    for d, r, c, rc, jq, ev in zip(docs, rel_norm, cit_norm, rec_norm, jq_norm, ev_raw):
        final = w_rel * r + w_cit * c + w_rec * rc + w_jq * jq + w_ev * ev
        d2 = dict(d)
        d2["final_score"] = final
        d2["final_score_components"] = {
            "relevance": r,
            "citations": c,
            "recency": rc,
            "journal_impact": jq,
            "evidence_level": ev,
        }
        out.append(d2)
    out.sort(key=lambda x: x["final_score"], reverse=True)
    return out

## Step 10 — run it and look at the ranking

In [ ]:
this_year = datetime.datetime.now().year

candidates = [
    {"title": "RCT of NPWT in diabetic foot ulcers", "rcs_score": 9, "citation_count": 240,
     "year": this_year - 3, "journal": "The Lancet", "evidence_level": "Level I"},
    {"title": "Meta-analysis of amputation risk reduction strategies", "rcs_score": 8, "citation_count": 512,
     "year": this_year - 6, "journal": "Diabetes Care", "evidence_level": "Level I"},
    {"title": "Cohort study of NPWT vs standard dressings", "rcs_score": 7, "citation_count": 34,
     "year": this_year - 1, "journal": "Wound Repair and Regeneration", "evidence_level": "Level II"},
    {"title": "Case series on pressure injury staging", "rcs_score": 6, "citation_count": 5,
     "year": this_year - 9, "journal": "PLOS ONE", "evidence_level": "Level IV"},
    {"title": "Expert consensus on offloading techniques", "rcs_score": 6, "citation_count": 18,
     "year": this_year - 4, "journal": "A Regional Wound Care Newsletter", "evidence_level": "Level V"},
    {"title": "Retrospective review of NPWT adoption patterns", "rcs_score": 5, "citation_count": 90,
     "year": this_year - 12, "journal": "Nature Medicine", "evidence_level": "Level III"},
    {"title": "Narrative review of diabetic foot ulcer epidemiology", "rcs_score": 5, "citation_count": 3,
     "year": this_year, "journal": "BMJ", "evidence_level": "Unknown"},
    {"title": "Single-arm pilot study of a novel dressing", "rcs_score": 6, "citation_count": 1,
     "year": this_year - 2, "journal": "A Regional Wound Care Newsletter", "evidence_level": "Level IV"},
]

ranked = quality_weighted_sort(candidates, DEFAULT_WEIGHTS, quality_map=journal_quality_map)
nbio.table(
    [
        (i + 1, f"{d['final_score']:.3f}", d["title"][:48])
        for i, d in enumerate(ranked)
    ],
    headers=("rank", "final_score", "title"),
)

In [ ]:
print("final_score_components for the top-ranked paper:")
nbio.show_json({"title": ranked[0]["title"], **ranked[0]["final_score_components"]})

## Step 11 — re-deriving a ranking from stored components — no retrieval, no API calls

This is the payoff. Treat the `final_score_components` already attached
above as if they had been persisted from a real run days or weeks ago.
`rerank_from_components` does not touch `rcs_score`, `citation_count`,
`year`, `journal`, or `evidence_level` again -- it reads only the five
already-normalized numbers and applies a new weight vector. An ablation is a
dictionary literal, not a pipeline run.

In [ ]:
def rerank_from_components(docs_with_components: list[dict], weights: dict) -> list[dict]:
    out = []
    for d in docs_with_components:
        comp = d["final_score_components"]
        final = sum(weights.get(k, 0.0) * v for k, v in comp.items())
        out.append({**d, "final_score": final})
    out.sort(key=lambda x: x["final_score"], reverse=True)
    return out


# A different, equally "neutral-example" weight vector: favor recency and evidence
# level over raw citation count -- e.g. for a reader who cares more about current,
# well-designed studies than about how many times a paper has been cited.
RECENCY_FIRST_WEIGHTS = {
    "relevance": 0.35,
    "citations": 0.05,
    "recency": 0.30,
    "journal_impact": 0.05,
    "evidence_level": 0.25,
}

reranked = rerank_from_components(ranked, RECENCY_FIRST_WEIGHTS)

rows = []
old_rank = {d["title"]: i + 1 for i, d in enumerate(ranked)}
for new_i, d in enumerate(reranked):
    rows.append((new_i + 1, old_rank[d["title"]], f"{d['final_score']:.3f}", d["title"][:44]))

nbio.table(rows, headers=("new rank", "old rank", "new final_score", "title"))

Compare the `new rank` / `old rank` columns above: several papers moved.
That reordering came entirely from `rerank_from_components` -- zero calls to
`quality_weighted_sort`, zero re-reads of `rcs_score` or `citation_count`,
zero network activity. The five persisted numbers were sufficient.

## Step 12 — seeing known defect 1 directly: what "journal_impact contributes 0.00" looks like

Simulate the real product's missing-CSV state -- an empty quality map, so
every `journal_quality_score` lookup returns `-1` and every `journal_impact`
component clamps to `0.0` -- and re-rank the same candidates under the exact
same `DEFAULT_WEIGHTS` used above. Nothing else changes; only the journal
signal goes dark.

In [ ]:
ranked_no_journal_data = quality_weighted_sort(candidates, DEFAULT_WEIGHTS, quality_map={})

jq_values = [d["final_score_components"]["journal_impact"] for d in ranked_no_journal_data]
print(f"journal_impact across all {len(jq_values)} candidates when the CSV is absent: {set(jq_values)}")
print(f"(the weight allocated to this signal, {DEFAULT_WEIGHTS['journal_impact']:.0%} of final_score, "
      f"is contributing exactly 0.0 to every paper -- silently, not as an error)\n")

rows = []
old_rank = {d["title"]: i + 1 for i, d in enumerate(ranked)}
for new_i, d in enumerate(ranked_no_journal_data):
    rows.append((new_i + 1, old_rank[d["title"]], f"{d['final_score']:.3f}", d["title"][:44]))
nbio.table(rows, headers=("rank w/o journal data", "rank w/ example CSV", "final_score", "title"))

## Wrap-up

`final_score_components` is the one field in this whole stage worth
insisting on keeping, even though it costs nothing to compute and nothing
obviously breaks if you drop it: without it, every ablation is a full
re-run through retrieval, dedup, and RCS; with it, an ablation is the two
cells above.

This closes out stage `04-retrieve`. `01` and `02` show the two retrieval
shapes; `03` and `04` show what happens to a candidate list before it's
ranked; `05` shows what a ranking actually is -- a weight vector applied to
five persisted numbers, nothing more exotic than that.